# Maia Lite 0.3.0 — Benchmark do modelo base (420 itens)

**Baseline pré-SFT** · PT / EN / ES · 150 fatos + 90 semântica + 90 aritmética + 90 compreensão contextual.

Baseado na **arquitetura exata** da célula 5 do notebook de pré-treinamento v3.8. Os pesos não são modificados. O benchmark JSONL acompanha este notebook e deve ser carregado em `/content` ou Drive.

**Limitações:** itens gerados/curados para diagnóstico, sem garantia de ausência de sobreposição com corpus de pré-treinamento; países/capitais podem apresentar viés de frequência; teste de compreensão usa exemplos sintéticos simples. Não é uma medida de capacidade geral. Não usar os itens de teste no SFT. Resultados devem ser comparados somente com o mesmo protocolo.

In [7]:
%pip -q install transformers pandas numpy matplotlib tqdm

## 1 — Caminhos, GPU e checkpoint

In [8]:
from pathlib import Path
import os, json, gc, hashlib, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from transformers import AutoTokenizer
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Fora do Colab: ajuste os caminhos abaixo.')

RUN_DIR = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/training_runs/maia_lite_0_3_0_a100_v3_experimental')
CHECKPOINT_PATH = RUN_DIR / 'latest.pt'
TOKENIZER_DIR = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/tokenizer')
BENCHMARK_PATH = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Benchmark_Base_420.jsonl')
OUTPUT_DIR = RUN_DIR / 'evaluation_base_420'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
assert CHECKPOINT_PATH.is_file(), CHECKPOINT_PATH
assert TOKENIZER_DIR.is_dir(), TOKENIZER_DIR
assert BENCHMARK_PATH.is_file(), 'Faça upload do JSONL para /content ou altere BENCHMARK_PATH.'


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda


## 2 — Arquitetura original do pré-treinamento v3.8 (sem alterações)

In [9]:
from dataclasses import dataclass
@dataclass
class Config:
    vocab_size:int=50257
    block_size:int=2048
    n_layer:int=36
    n_head:int=16
    n_embd:int=1024
    dropout:float=0.0
    bias:bool=True
cfg=Config()
class Block(nn.Module):
    def __init__(self,c):
        super().__init__()
        d=c.n_embd
        self.ln1=nn.LayerNorm(d)
        self.qkv=nn.Linear(d,3*d,bias=c.bias)
        self.proj=nn.Linear(d,d,bias=c.bias)
        self.ln2=nn.LayerNorm(d)
        self.fc=nn.Linear(d,4*d,bias=c.bias)
        self.out=nn.Linear(4*d,d,bias=c.bias)
        self.nh=c.n_head
    def forward(self,x):
        b,t,d=x.shape
        q,k,v=self.qkv(self.ln1(x)).chunk(3,dim=-1)
        q=q.view(b,t,self.nh,d//self.nh).transpose(1,2)
        k=k.view(b,t,self.nh,d//self.nh).transpose(1,2)
        v=v.view(b,t,self.nh,d//self.nh).transpose(1,2)
        a=F.scaled_dot_product_attention(q,k,v,is_causal=True,dropout_p=0.0)
        x=x+self.proj(a.transpose(1,2).contiguous().view(b,t,d))
        x=x+self.out(F.gelu(self.fc(self.ln2(x)),approximate='tanh'))
        return x
class MaiaGPT(nn.Module):
    def __init__(self,c):
        super().__init__();self.cfg=c
        self.tok=nn.Embedding(c.vocab_size,c.n_embd)
        self.pos=nn.Embedding(c.block_size,c.n_embd)
        self.blocks=nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.norm=nn.LayerNorm(c.n_embd)
        self.head=nn.Linear(c.n_embd,c.vocab_size,bias=False)
        self.head.weight=self.tok.weight
        self.apply(self.init_weights)
    @staticmethod
    def init_weights(m):
        if isinstance(m,(nn.Linear,nn.Embedding)):
            nn.init.normal_(m.weight,mean=0,std=0.02)
            if isinstance(m,nn.Linear) and m.bias is not None:nn.init.zeros_(m.bias)
    def forward(self,idx,targets=None):
        b,t=idx.shape
        assert t<=self.cfg.block_size
        x=self.tok(idx)+self.pos(torch.arange(t,device=idx.device))
        for block in self.blocks:x=block(x)
        x=self.norm(x)
        logits=self.head(x)
        loss=None if targets is None else F.cross_entropy(logits.reshape(-1,logits.size(-1)).float(),targets.reshape(-1),ignore_index=-100)
        return logits,loss


## 3 — Carregamento do tokenizer e checkpoint (somente pesos na GPU)

`weights_only=False` é necessário para o checkpoint com estados RNG/otimizador; **execute somente com checkpoint de origem confiável**. A leitura inicial em CPU pode exigir RAM adicional.

In [10]:
tokenizer = AutoTokenizer.from_pretrained(str(TOKENIZER_DIR), local_files_only=True, use_fast=True, trust_remote_code=False)
assert len(tokenizer) == cfg.vocab_size
assert tokenizer.eos_token_id == 2 and tokenizer.pad_token_id == 1
print('Tokenizer:',len(tokenizer),'EOS:',tokenizer.eos_token_id)

ck = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
step = ck['step']; identity = ck['identity']
model = MaiaGPT(cfg)
model.load_state_dict(ck['model'], strict=True)
del ck
gc.collect()
model.to(DEVICE).eval()
assert model.head.weight.data_ptr() == model.tok.weight.data_ptr()
print('Step:',step,'Parâmetros:',sum(p.numel() for p in model.parameters()))


Tokenizer: 50257 EOS: 2
Step: 300000 Parâmetros: 507026432


## 4 — Ler e validar benchmark (420 questões, três idiomas)

In [11]:
with open(BENCHMARK_PATH, encoding='utf-8') as f:
    items = [json.loads(line) for line in f if line.strip()]
assert len(items)==420
assert len({x['id'] for x in items})==420
assert {x['language'] for x in items}=={'pt','en','es'}
assert {x['category'] for x in items}=={'factual','semantic','math','reading'}
for x in items:
    assert len(x['options'])==3 and len(set(x['options']))==3
    assert 0 <= x['correct_index'] < 3
print(pd.crosstab(pd.Series([x['category'] for x in items],name='categoria'),pd.Series([x['language'] for x in items],name='idioma')))


idioma     en  es  pt
categoria            
factual    50  50  50
math       30  30  30
reading    30  30  30
semantic   30  30  30


## 5 — Scoring autoregressivo por log-verossimilhança

Usa a probabilidade condicional da **sequência completa de tokens** e registra `logp_sum`, `logp_mean`, `token_count`. A métrica primária é o maior `logp_sum`; `logp_mean` é diagnóstico e não elimina vieses de comprimento. Exclui itens cuja tokenização do prefixo não é preservada. Para comparar alternativas, os logits do prefixo são cacheados apenas logicamente (sem KV cache); não há truncamento silencioso.

In [12]:
@torch.inference_mode()
def score_choice(prefix, suffix):
    prefix_ids = tokenizer.encode(prefix, add_special_tokens=False)
    full_ids = tokenizer.encode(prefix + suffix, add_special_tokens=False)
    if not prefix_ids or full_ids[:len(prefix_ids)] != prefix_ids:
        return None, 'token_boundary_mismatch'
    if len(full_ids) > cfg.block_size:
        return None, 'context_overflow'
    x = torch.tensor([full_ids[:-1]],dtype=torch.long,device=DEVICE)
    logits,_ = model(x)
    start=len(prefix_ids)-1
    targets=torch.tensor(full_ids[len(prefix_ids):],dtype=torch.long,device=DEVICE)
    lp=F.log_softmax(logits[0,start:start+len(targets)].float(),dim=-1)
    scores=lp.gather(1,targets[:,None]).squeeze(1)
    return {'logp_sum':float(scores.sum().item()),'logp_mean':float(scores.mean().item()),'token_count':len(targets),'first_token_logp':float(scores[0].item())},None

def evaluate_item(item):
    scores=[]; errors=[]
    for suffix in item['options']:
        result,err=score_choice(item['prefix'],suffix)
        scores.append(result);errors.append(err)
    row={k:item[k] for k in ['id','category','language','fact_id','prefix','correct_index']}
    row['options']=json.dumps(item['options'],ensure_ascii=False)
    row['errors']=json.dumps(errors,ensure_ascii=False)
    row['valid']=all(s is not None for s in scores)
    if row['valid']:
        lp=[s['logp_sum'] for s in scores]
        mean=[s['logp_mean'] for s in scores]
        best=max(range(3),key=lambda i:lp[i])
        row.update(predicted_index=best,correct=int(best==item['correct_index']),
                   margin_correct_vs_best_wrong=lp[item['correct_index']]-max(lp[i] for i in range(3) if i!=item['correct_index']),
                   top_margin=sorted(lp,reverse=True)[0]-sorted(lp,reverse=True)[1],
                   logp_sum=json.dumps(lp),logp_mean=json.dumps(mean),
                   token_counts=json.dumps([s['token_count'] for s in scores]),
                   first_token_logp=json.dumps([s['first_token_logp'] for s in scores]),
                   predicted_mean_index=max(range(3),key=lambda i:mean[i]))
    return row

# Teste de integridade de scoring
probe=evaluate_item(items[0]);print(probe)


{'id': 'factual_000_pt', 'category': 'factual', 'language': 'pt', 'fact_id': 'factual_000', 'prefix': 'A capital de Brasil é', 'correct_index': 1, 'options': '[" Rio de Janeiro.", " Brasília.", " São Paulo."]', 'errors': '[null, null, null]', 'valid': True, 'predicted_index': 2, 'correct': 0, 'margin_correct_vs_best_wrong': -0.37325048446655273, 'top_margin': 0.37325048446655273, 'logp_sum': '[-6.402182579040527, -5.4321136474609375, -5.058863162994385]', 'logp_mean': '[-1.6005456447601318, -2.7160568237304688, -1.686287760734558]', 'token_counts': '[4, 2, 3]', 'first_token_logp': '[-5.259452819824219, -4.306218147277832, -3.8501179218292236]', 'predicted_mean_index': 0}


## 6 — Executar 420 questões

A avaliação é sequencial e pode levar tempo; cada item faz três passes. O processo grava resultados parciais a cada 15 questões, permitindo retomar sem repetir os itens concluídos. Para evitar misturar checkpoints diferentes, o nome do arquivo inclui o `step`.

In [13]:
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
RESULT_CSV=OUTPUT_DIR/f'benchmark_420_step_{step}.csv'
if RESULT_CSV.exists():
    previous=pd.read_csv(RESULT_CSV)
    completed=set(previous['id'].tolist())
    records=previous.to_dict('records')
    print('Retomando:',len(records),'itens')
else:
    records=[];completed=set()
start=time.time()
for item in tqdm(items,desc='Benchmark 420'):
    if item['id'] in completed:continue
    records.append(evaluate_item(item))
    if len(records)%15==0:
        pd.DataFrame(records).to_csv(RESULT_CSV,index=False)
pd.DataFrame(records).to_csv(RESULT_CSV,index=False)
results=pd.DataFrame(records)
print('Concluídos:',len(results),'Válidos:',int(results['valid'].sum()),'Tempo (s):',round(time.time()-start,1))
print('CSV:',RESULT_CSV)


Benchmark 420:   0%|          | 0/420 [00:00<?, ?it/s]

Concluídos: 420 Válidos: 420 Tempo (s): 16.3
CSV: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/training_runs/maia_lite_0_3_0_a100_v3_experimental/evaluation_base_420/benchmark_420_step_300000.csv


## 7 — Relatório de acurácia, margens e erros

In [14]:
valid=results[results['valid']==True].copy()
if valid.empty:
    raise RuntimeError('Nenhum item válido. Inspecione errors e tokenização.')
summary=valid.groupby(['category','language']).agg(n=('correct','size'),accuracy=('correct','mean'),mean_margin=('margin_correct_vs_best_wrong','mean')).reset_index()
print(summary.to_string(index=False))
print('\nAcurácia global válida:',round(valid['correct'].mean(),4),'n=',len(valid))
print('Cobertura:',round(len(valid)/len(items),4))
print('\nErros e exclusões:')
print(results.loc[~results['valid'],['id','errors']].head(20).to_string(index=False))
summary.to_csv(OUTPUT_DIR/f'summary_step_{step}.csv',index=False)


category language  n  accuracy  mean_margin
 factual       en 50  0.920000     3.152361
 factual       es 50  0.800000     1.774948
 factual       pt 50  0.780000     1.848520
    math       en 30  0.433333    -0.122907
    math       es 30  0.433333    -0.122907
    math       pt 30  0.433333    -0.122907
 reading       en 30  0.966667     1.240964
 reading       es 30  1.000000     0.724320
 reading       pt 30  0.633333     0.430263
semantic       en 30  0.666667     2.142847
semantic       es 30  0.766667     2.028568
semantic       pt 30  0.700000     2.146219

Acurácia global válida: 0.7286 n= 420
Cobertura: 1.0

Erros e exclusões:
Empty DataFrame
Columns: [id, errors]
Index: []


## 8 — Diagnóstico de sensibilidade e confiabilidade

O conjunto contém um enunciado por fato em cada idioma; portanto a sensibilidade aqui mede **consistência entre idiomas** e não paráfrases dentro do mesmo idioma. Para medir paráfrases, deve-se criar um conjunto adicional de variações mantendo os mesmos fatos e respostas.

In [15]:
fact=valid[valid.category=='factual'].copy()
consistency=fact.groupby('fact_id').agg(n_languages=('language','nunique'),all_correct=('correct','min'),mean_accuracy=('correct','mean')).reset_index()
complete=consistency[consistency.n_languages==3]
print('Fatos avaliados nos 3 idiomas:',len(complete))
print('Todos os idiomas corretos:',int(complete.all_correct.sum()),'de',len(complete))
print('Acurácia por idioma:\n',fact.groupby('language')['correct'].agg(['mean','count']))
print('\nExemplos de erros:')
print(valid.loc[valid.correct==0,['id','prefix','options','predicted_index','correct_index','margin_correct_vs_best_wrong']].head(20).to_string(index=False))


Fatos avaliados nos 3 idiomas: 50
Todos os idiomas corretos: 36 de 50
Acurácia por idioma:
           mean  count
language             
en        0.92     50
es        0.80     50
pt        0.78     50

Exemplos de erros:
            id                      prefix                                              options  predicted_index  correct_index  margin_correct_vs_best_wrong
factual_000_pt       A capital de Brasil é    [" Rio de Janeiro.", " Brasília.", " São Paulo."]                2              1                     -0.373250
factual_000_en    The capital of Brazil is    [" Brasília.", " São Paulo.", " Rio de Janeiro."]                2              0                     -0.907346
factual_000_es     La capital de Brasil es    [" Brasília.", " São Paulo.", " Rio de Janeiro."]                1              0                     -2.829709
factual_008_pt       A capital de Canadá é             [" Vancouver.", " Ottawa.", " Toronto."]                2              1                   

## 9 — Perplexidade (opcional, exige corpus held-out comprovadamente independente)

**Não** calculamos perplexidade sobre as próprias questões nem inventamos dados de validação. Defina `HELDOUT_TEXT_PATH` para um `.txt` que não tenha sido usado no pré-treinamento nem no SFT. O cálculo usa blocos não sobrepostos de até 2048 tokens e soma perdas de todos os tokens previstos, exceto o primeiro de cada bloco.

In [16]:
HELDOUT_TEXT_PATH = None  # exemplo: '/content/heldout_pt.txt'
@torch.inference_mode()
def heldout_perplexity(path, max_tokens=30000):
    text=Path(path).read_text(encoding='utf-8')
    ids=tokenizer.encode(text,add_special_tokens=False)[:max_tokens]
    assert len(ids)>=2
    total_nll=0.0;total_count=0
    for start in tqdm(range(0,len(ids)-1,cfg.block_size),desc='Heldout PPL'):
        chunk=ids[start:start+cfg.block_size]
        if len(chunk)<2:continue
        x=torch.tensor([chunk[:-1]],dtype=torch.long,device=DEVICE)
        y=torch.tensor(chunk[1:],dtype=torch.long,device=DEVICE)
        logits,_=model(x)
        nll=F.cross_entropy(logits[0].float(),y,reduction='sum')
        total_nll+=float(nll.item());total_count+=len(chunk)-1
    return {'tokens_scored':total_count,'mean_nll':total_nll/total_count,'perplexity':float(__import__('math').exp(total_nll/total_count))}
if HELDOUT_TEXT_PATH:
    print(heldout_perplexity(HELDOUT_TEXT_PATH))
else:
    print('Perplexidade não calculada: corpus held-out não fornecido.')


Perplexidade não calculada: corpus held-out não fornecido.


## 10 — Manifesto e comparação futura com SFT

Congele este JSONL e seu SHA-256. Rode o mesmo notebook com pesos SFT, mantendo tokenizer, prompt, opções e regra de scoring. **Não treine com este conjunto**. Não interprete acurácia em 420 itens como medida representativa sem avaliação externa independente.

In [17]:
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b''):h.update(chunk)
    return h.hexdigest()
manifest={'project':'Maia Lite 0.3.0','evaluation':'base_420_v1','step':int(step),
          'checkpoint_path':str(CHECKPOINT_PATH),'tokenizer_path':str(TOKENIZER_DIR),
          'benchmark_sha256':sha256_file(BENCHMARK_PATH),'benchmark_items':len(items),
          'scoring':'sum_logp_conditional_full_completion','valid_items':int(results['valid'].sum()),
          'device':str(DEVICE),'torch':torch.__version__,
          'note':'Benchmark diagnóstico; possível sobreposição com pré-treinamento; não usar para SFT.'}
manifest_path=OUTPUT_DIR/f'manifest_step_{step}.json'
manifest_path.write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(manifest,ensure_ascii=False,indent=2))
print('Relatórios:',OUTPUT_DIR)


{
  "project": "Maia Lite 0.3.0",
  "evaluation": "base_420_v1",
  "step": 300000,
  "checkpoint_path": "/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/training_runs/maia_lite_0_3_0_a100_v3_experimental/latest.pt",
  "tokenizer_path": "/content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/tokenizer",
  "benchmark_sha256": "04c0155dc61c5a7df34732112d04915566143fd0226efa6d641c6d4f43480794",
  "benchmark_items": 420,
  "scoring": "sum_logp_conditional_full_completion",
  "valid_items": 420,
  "device": "cuda",
  "torch": "2.14.1+cu130",
  "note": "Benchmark diagnóstico; possível sobreposição com pré-treinamento; não usar para SFT."
}
Relatórios: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/training_runs/maia_lite_0_3_0_a100_v3_experimental/evaluation_base_420


In [18]:
from pathlib import Path

REPORT_DIR = Path(
    "/content/drive/MyDrive/Colab_LLMs/"
    "Maia_Lite_0_3_0_Chat/training_runs/"
    "maia_lite_0_3_0_a100_v3_experimental/"
    "evaluation_base_420"
)

for p in sorted(REPORT_DIR.rglob("*")):
    if p.is_file():
        print(f"{p.relative_to(REPORT_DIR)} — {p.stat().st_size:,} bytes")

benchmark_420_step_300000.csv — 161,354 bytes
manifest_step_300000.json — 645 bytes
summary_step_300000.csv — 594 bytes
